In [1]:
import pandas as pd
from datetime import date
import calendar
from utils import snowflake_login, carga_snow_generic, descargar_segmento, get_credentials
import gspread
from gspread_dataframe import get_as_dataframe
from google.oauth2.service_account import Credentials
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
import re
import streamlit as st
import numpy as np

In [2]:
#Funciones de redondeo
def red (r:float):
    allowed_numbers = [0.2, 0.25, 0.4, 0.5, 0.6, 0.75, 0.8, 1.0, 1.2, 1.25, 1.4, 1.5, 1.6, 1.75, 1.8, 2.0,
                       2.2, 2.25, 2.4, 2.5, 2.6, 2.75, 2.8, 3.0, 3.2, 3.25, 3.4, 3.5, 3.6, 3.75, 3.8, 4.0]
    closest_number = min(allowed_numbers, key=lambda x: abs(x - r))
    return closest_number

def red3 (r:float):
    allowed_numbers = [1.0, 2.0, 3.0, 4.0]
    closest_number = min(allowed_numbers, key=lambda x: abs(x - r))
    return closest_number

def red7 (r:float):
    allowed_numbers = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
    closest_number = min(allowed_numbers, key=lambda x: abs(x - r))
    return closest_number

#Funcion para cerrar la participación
def cerrar (df:pd.DataFrame):
    if df['PARTICIPACION'].sum() > 5.1:
        if df.shape[0] <= 3:
            df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                 ascending=False).head(1)['ITEM'].values[0]].index,
                   'PARTICIPACION'] = df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                                           ascending=False).head(1)['ITEM'].values[0]].index,
                                             'PARTICIPACION'] - 1
            df = cerrar(df)
            return df
        elif df.shape[0] <= 7:
            df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                 ascending=False).head(1)['ITEM'].values[0]].index,
                   'PARTICIPACION'] = df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                                           ascending=False).head(1)['ITEM'].values[0]].index,
                                             'PARTICIPACION'] - 0.5
            df = cerrar(df)
            return df
        else:
            df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                 ascending=False).head(1)['ITEM'].values[0]].index,
                   'PARTICIPACION'] = df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                                           ascending=False).head(1)['ITEM'].values[0]].index,
                                             'PARTICIPACION'] - 0.2
            df = cerrar(df)
            return df
    elif df['PARTICIPACION'].sum() < 4.9:
        if df.shape[0] <= 3:
            df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                 ascending=False).head(1)['ITEM'].values[0]].index,
                   'PARTICIPACION'] = df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                                           ascending=False).head(1)['ITEM'].values[0]].index,
                                             'PARTICIPACION'] + 1
            df = cerrar(df)
            return df
        elif df.shape[0] <= 7:
            df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                 ascending=False).head(1)['ITEM'].values[0]].index,
                   'PARTICIPACION'] = df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                                           ascending=False).head(1)['ITEM'].values[0]].index,
                                             'PARTICIPACION'] + 0.5
            df = cerrar(df)
            return df
        else:
            df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                 ascending=False).head(1)['ITEM'].values[0]].index,
                   'PARTICIPACION'] = df.loc[df[df['ITEM']==df.sort_values('PARTICIPACION',
                                                                           ascending=False).head(1)['ITEM'].values[0]].index,
                                             'PARTICIPACION'] + 0.2
            df = cerrar(df)
            return df
    else:
        return df

#Función para obtener la participación
def participacion (cursor, punt:pd.DataFrame, fecha_inicio:str):
    #Descargamos df
    dim = descargar_segmento(cursor=cursor, query='Dimensiones',
                             cond=f" and ip.fecha_inicio = '{fecha_inicio}';")
    ac = descargar_segmento(cursor=cursor, query='Aceleraciones').astype('str').replace('nan', pd.NA)

    #Cambiamos formato de columnas
    dim = dim.astype({'ITEM':'str', 'LOCAL':'str', 'ITEMS_POR_FRENTE':'int64',
                      'ITEMS_POR_LATERAL':'int64', 'CANT_MAX_X_ESTANTE':'int64', 'CANT_MIN_X_ESTANTE':'int64'})
    ac = ac.astype({'ACELERACION_ART':'float64', 'ACELERACION_SUBCLASE':'float64', 'ACELERACION_CLASE':'float64'})
    
    #Armamos df principal
    cab = dim.merge(ac[['LOCAL', 'ITEM', 'ACELERACION_ART']],
                    how='left').merge(ac[['LOCAL', 'SUBCLASE', 'ACELERACION_SUBCLASE']],
                                      how='left').merge(ac[['LOCAL', 'CLASE', 'ACELERACION_CLASE']],
                                                        how='left').drop_duplicates().reset_index(drop=True)
    #Definimos aceleración
    cab['ACELERACION'] = cab['ACELERACION_ART']
    cab.loc[cab[cab['ACELERACION'].isna()].index,
            'ACELERACION'] = cab.loc[cab[cab['ACELERACION'].isna()].index, 'ACELERACION_SUBCLASE']
    cab.loc[cab[cab['ACELERACION'].isna()].index,
            'ACELERACION'] = cab.loc[cab[cab['ACELERACION'].isna()].index, 'ACELERACION_CLASE']
    
    #Donde no tengamos dato de aceleración o donde esta sea menor a 1 asumimos que no se acelera
    cab.fillna({'ACELERACION':1}, inplace=True)
    cab.loc[cab[cab['ACELERACION']<1].index, 'ACELERACION'] = 1

    #Definimos unidades diarias y DOH
    cab['UNIDADES_DIARIAS_POR_ACELERACION'] = cab['AVG_BASAL_180']*cab['ACELERACION']
    cab['DOH_ESTANTE_MIN'] = round(cab['CANT_MIN_X_ESTANTE']/cab['UNIDADES_DIARIAS_POR_ACELERACION'])
    cab['DOH_ESTANTE_MAX'] = round(cab['CANT_MAX_X_ESTANTE']/cab['UNIDADES_DIARIAS_POR_ACELERACION'])

    #Limpiamos un poco el df de cabeceras
    cab.drop(index=cab[~cab['ITEM'].isin(punt['ITEM'].unique())].index, inplace=True)

    #Tomamos columnas que interesan
    cab = cab[['LOCAL', 'NOMBRE_TIENDA', 'GRUPO', 'CLASE', 'SUBCLASE', 'ITEM', 'ARTC_ARTC_DESC', 'AVG_BASAL_180',
               'ACELERACION', 'UNIDADES_DIARIAS_POR_ACELERACION', 'PROFUNDIDAD', 'FRENTE', 'ALTURA',
               'CANT_MAX_X_ESTANTE', 'CANT_MIN_X_ESTANTE', 'DOH_ESTANTE_MIN', 'DOH_ESTANTE_MAX']]

    #Algoritmo que define la participación por puntera
    df_final = pd.DataFrame()
    for l in punt['LOCAL'].unique():
        df = punt[(punt['LOCAL']==l)].drop_duplicates()
        for p in df['PUNTERA'].unique():
            try:
                df = punt[(punt['LOCAL']==l) &
                          (punt['PUNTERA']==p)].drop(columns='DESCRIPCION').merge(cab).drop_duplicates()
                df['PARTICIPACION'] = np.NaN
                if df[df['DOH_ESTANTE_MAX'] <= 31].shape[0] >= 5:
                    #print('ESTANTES INSUFICIENTES')
                    if df.shape[0] >= 7:
                        it_may_vta = df.sort_values('DOH_ESTANTE_MAX').head(3)['ITEM'].values
                        it_h_vta = df.sort_values('DOH_ESTANTE_MAX').head()['ITEM'].values
                        df.loc[df[df['ITEM'].isin(it_may_vta)].index, 'PARTICIPACION'] = 1
                        df.loc[df[(~df['ITEM'].isin(it_may_vta)) &
                                    (df['ITEM'].isin(it_h_vta))].index, 'PARTICIPACION'] = 0.5
                        df.loc[df[~df['ITEM'].isin(it_h_vta)].index,
                                'PARTICIPACION'] = 1/df[~df['ITEM'].isin(it_h_vta)].shape[0]
                    elif df.shape[0] == 6:
                        it_may_vta = df.sort_values('DOH_ESTANTE_MAX').head(4)['ITEM'].values
                        df.loc[df[df['ITEM'].isin(it_may_vta)].index, 'PARTICIPACION'] = 1
                        df.loc[df[(~df['ITEM'].isin(it_may_vta))].index, 'PARTICIPACION'] = 0.5
                    else:
                        df.loc[df.index, 'PARTICIPACION'] = 1
                elif df.shape[0] == 1:
                    df['PARTICIPACION'] = 5
                elif df.shape[0] <= 3:
                    t = df['DOH_ESTANTE_MAX'].sum()
                    df['PARTICIPACION'] = t/df['DOH_ESTANTE_MAX']
                    t = df['PARTICIPACION'].sum()
                    df['PARTICIPACION'] = 5*df['PARTICIPACION']/t
                    df['PARTICIPACION'] = df['PARTICIPACION'].apply(red3)
                elif df.shape[0] <= 7:
                    t = df['DOH_ESTANTE_MAX'].sum()
                    df['PARTICIPACION'] = t/df['DOH_ESTANTE_MAX']
                    t = df['PARTICIPACION'].sum()
                    df['PARTICIPACION'] = 5*df['PARTICIPACION']/t
                    df['PARTICIPACION'] = df['PARTICIPACION'].apply(red7)
                else:
                    t = df['DOH_ESTANTE_MAX'].sum()
                    df['PARTICIPACION'] = t/df['DOH_ESTANTE_MAX']
                    t = df['PARTICIPACION'].sum()
                    df['PARTICIPACION'] = 5*df['PARTICIPACION']/t
                    df['PARTICIPACION'] = df['PARTICIPACION'].apply(red)
                df = cerrar(df=df)
                df_final = pd.concat([df_final, df], ignore_index=True)
            except:
                pass
    
    #Vemos los items que no teniamos datos de aceleración/dimensiones
    df_final['DURACION'] = (pd.to_datetime(df_final['FECHA_FIN']) -
                            pd.to_datetime(df_final['FECHA_INICIO'])).mean().days + 1
    #Agregamos los faltantes
    df_final['UNIDADES_CARGA_POR_VENTA'] = df_final['DURACION']*df_final['UNIDADES_DIARIAS_POR_ACELERACION']
    df_final['UNIDADES_CARGA_MAX_POR_PARTICIPACION'] = df_final['PARTICIPACION']*df_final['CANT_MAX_X_ESTANTE']
    df_final['UNIDADES_CARGA_MIN_POR_PARTICIPACION'] = df_final['PARTICIPACION']*df_final['CANT_MIN_X_ESTANTE']

    df_final = df_final[['FECHA_INICIO', 'FECHA_FIN', 'SECCION', 'LOCAL', 'NOMBRE_TIENDA', 'PUNTERA',
                         'GRUPO', 'CLASE', 'SUBCLASE', 'ESTADISTICO', 'ITEM', 'ARTC_ARTC_DESC',
                         'AVG_BASAL_180', 'ACELERACION', 'UNIDADES_DIARIAS_POR_ACELERACION',
                         'PROFUNDIDAD', 'FRENTE', 'ALTURA', 'CANT_MAX_X_ESTANTE', 'CANT_MIN_X_ESTANTE',
                         'DOH_ESTANTE_MIN', 'DOH_ESTANTE_MAX', 'PARTICIPACION', 'DURACION', 'UNIDADES_CARGA_POR_VENTA',
                         'UNIDADES_CARGA_MAX_POR_PARTICIPACION', 'UNIDADES_CARGA_MIN_POR_PARTICIPACION']]
    return df_final

In [3]:
scopes = ['https://www.googleapis.com/auth/spreadsheets',
          'https://www.googleapis.com/auth/drive']

# keyboard.press_and_release('ctrl+w')        #Close the window

credentials = Credentials.from_service_account_file('api_credentials_2.json', scopes=scopes)

gc = gspread.authorize(credentials)
gauth = GoogleAuth()
drive = GoogleDrive(gauth)

In [4]:
def renombrar_columnas(col_name):
    if re.search(r'EXHIBI', col_name, re.IGNORECASE):
        return f'PUNTERA'
    elif re.search(r'^ESTAD', col_name, re.IGNORECASE):
        return f'ESTADISTICO'
    elif re.search(r'^TEM', col_name, re.IGNORECASE):
        return f'ITEM'
    elif re.search(r'^Descripc', col_name, re.IGNORECASE):
        return f'DESCRIPCION'
    else:
        return col_name

In [5]:
url = 'https://docs.google.com/spreadsheets/d/1yME79mb27tG3FNLton-2pgEMH7oRjqXZMDWtwoQP-qQ/edit?pli=1&gid=350335113#gid=350335113'
sheet = url.split('/')[-2]
gs = gc.open_by_key(sheet)

In [7]:
meses = {"Enero": 1, "Febrero": 2, "Marzo": 3, "Abril": 4, "Mayo": 5, "Junio": 6, "Julio": 7,
            "Agosto": 8, "Septiembre": 9, "Octubre": 10, "Noviembre": 11, "Diciembre": 12}
mes = 'Agosto'
fecha = date.today().replace(month=meses[mes])
fecha

datetime.date(2026, 8, 13)

In [8]:
if fecha <= date.today():
    fecha = fecha.replace(year=date.today().year + 1)

In [9]:
fecha_inicio = fecha.replace(day=1).strftime('%Y-%m-%d')
fecha_fin = fecha.replace(day=calendar.monthrange(date.today().replace(month=fecha.month).year,
                                                date.today().replace(month=fecha.month).month)[1]).strftime('%Y-%m-%d')
print('Fecha inicio: ' + fecha_inicio)
print('Fecha fin: ' + fecha_fin)

Fecha inicio: 2026-08-01
Fecha fin: 2026-08-31


In [10]:
df_total = pd.DataFrame()

# Obtener todas las hojas en el Google Sheet
hojas = gs.worksheets()

# Iterar sobre cada hoja
for hoja in hojas:
    # Convertir la hoja actual a DataFrame
    worksheetL = gs.worksheet(hoja.title)
    juli0 = get_as_dataframe(worksheetL, header=None)
    
    # Encontrar la fila que contiene la palabra "EXHIBICION" en la primera columna
    col_0 = juli0.iloc[:, 0].astype(str)  # Ensure column 0 is all strings

    mask = col_0.str.contains("EXHIBICIÓN", na=False)

    condicion = juli0[mask].index[0] if mask.any() else None

    if condicion is not None:
        # Filtrar las filas a partir de la condición
        df = juli0.loc[condicion:, 0:5]
        df.columns = df.iloc[0]  # Definir la primera fila como encabezado
        df = df[1:].reset_index(drop=True)  # Reiniciar el índice
        
        # Renombrar columnas del DataFrame
        df.rename(columns=renombrar_columnas, inplace=True)

        df = df.dropna(subset=['PUNTERA'])
        
        # Limpiar columnas antes de convertir a enteros
        for col in ['ESTADISTICO', 'ITEM', 'LOCAL']:
            df[col] = df[col].astype(str)
            df[col] = df[col].str.replace('.', '', regex=False).fillna('0')
            df[col] = df[col].str.replace('nan', '0', regex=False).fillna('0')
            #df[col] = df[col].astype(int)
        df['SECCION'] = hoja.title

        # Concatenar el DataFrame actual con el total
        df_total = pd.concat([df_total, df], ignore_index=True).drop_duplicates()

#Definimos las columnas que nos interesan
df_total = df_total[['PUNTERA', 'ESTADISTICO', 'ITEM', 'DESCRIPCION', 'LOCAL', 'SECCION']]
df_total

,PUNTERA,ESTADISTICO,ITEM,DESCRIPCION,LOCAL,SECCION
0,CABECERA VAN DAM,261509000,1000055471,BOMBONERA LECHE BON O BON 270 G,101,Almacen
1,CABECERA VAN DAM,261509000,1000055471,BOMBONERA LECHE BON O BON 270 G,111,Almacen
2,CABECERA VAN DAM,261509000,1000055471,BOMBONERA LECHE BON O BON 270 G,115,Almacen
3,CABECERA VAN DAM,261509000,1000055471,BOMBONERA LECHE BON O BON 270 G,125,Almacen
4,CABECERA VAN DAM,261509000,1000055471,BOMBONERA LECHE BON O BON 270 G,126,Almacen
...,...,...,...,...,...,...
13291,CABECERAS,330805000,1000469754,SOFRITO FRUTOS DEL MAIPO 150 G,101,Congelados
13292,CABECERAS,330177000,1000059137,HAMBURGUESA CLASICA LA DOLFINA 150 GRS 3.00 U,101,Congelados
13293,CABECERAS,331636000,1000059548,HAMBURGUESA DE POLLO REB SADINESA 1.00 U,101,Congelados
13294,EQUIPO SADIA,331636000,1000059548,HAMBURGUESA DE POLLO REBOZADA SADINESA 1 UD,124,Congelados


In [11]:
# Mostrar las primeras filas del DataFrame total
print('Cantidad de registros: ' + str(df_total.shape[0]))
df_total['FECHA_INICIO'] = fecha_inicio
df_total['FECHA_FIN'] = fecha_fin
df_total

Cantidad de registros: 13296


,PUNTERA,ESTADISTICO,ITEM,DESCRIPCION,LOCAL,SECCION,FECHA_INICIO,FECHA_FIN
0,CABECERA VAN DAM,261509000,1000055471,BOMBONERA LECHE BON O BON 270 G,101,Almacen,2026-08-01,2026-08-31
1,CABECERA VAN DAM,261509000,1000055471,BOMBONERA LECHE BON O BON 270 G,111,Almacen,2026-08-01,2026-08-31
2,CABECERA VAN DAM,261509000,1000055471,BOMBONERA LECHE BON O BON 270 G,115,Almacen,2026-08-01,2026-08-31
3,CABECERA VAN DAM,261509000,1000055471,BOMBONERA LECHE BON O BON 270 G,125,Almacen,2026-08-01,2026-08-31
4,CABECERA VAN DAM,261509000,1000055471,BOMBONERA LECHE BON O BON 270 G,126,Almacen,2026-08-01,2026-08-31
...,...,...,...,...,...,...,...,...
13291,CABECERAS,330805000,1000469754,SOFRITO FRUTOS DEL MAIPO 150 G,101,Congelados,2026-08-01,2026-08-31
13292,CABECERAS,330177000,1000059137,HAMBURGUESA CLASICA LA DOLFINA 150 GRS 3.00 U,101,Congelados,2026-08-01,2026-08-31
13293,CABECERAS,331636000,1000059548,HAMBURGUESA DE POLLO REB SADINESA 1.00 U,101,Congelados,2026-08-01,2026-08-31
13294,EQUIPO SADIA,331636000,1000059548,HAMBURGUESA DE POLLO REBOZADA SADINESA 1 UD,124,Congelados,2026-08-01,2026-08-31


In [12]:
print('Articulos repetidos en puntera:')
rep = df_total[['LOCAL', 'PUNTERA', 'ITEM']]
df_total.loc[rep[rep.duplicated(keep=False)].index].sort_values(['LOCAL', 'PUNTERA', 'ITEM'])

Articulos repetidos en puntera:


,PUNTERA,ESTADISTICO,ITEM,DESCRIPCION,LOCAL,SECCION,FECHA_INICIO,FECHA_FIN
13294,EQUIPO SADIA,331636000,1000059548,HAMBURGUESA DE POLLO REBOZADA SADINESA 1 UD,124,Congelados,2026-08-01,2026-08-31
13295,EQUIPO SADIA,331636000,1000059548,HAMBURGUESA DE POLLO REBOZADA SADINESA 1 UD,124,Congelados,2026-08-01,2026-08-31


In [14]:
print('Articulos en varias punteras de un solo local:')
rep = df_total[['LOCAL', 'ITEM']]
df_total.loc[rep[rep.duplicated(keep=False)].index].sort_values(['LOCAL', 'ITEM'])

Articulos en varias punteras de un solo local:


,PUNTERA,ESTADISTICO,ITEM,DESCRIPCION,LOCAL,SECCION,FECHA_INICIO,FECHA_FIN
5790,CABECERA COLGATE ANUAL,160216000,1000037642,CREMA DENTAL COLGATE TOTAL 12 WHITENING 90 G,101,PyL,2026-08-01,2026-08-31
6042,LATERAL CHICO,160216000,1000037642,CREMA DENTAL COLGATE TOTAL 12 WHITENING 90 G,101,PyL,2026-08-01,2026-08-31
5791,CABECERA COLGATE ANUAL,160288000,1000037689,CREMA DENTAL COLGATE CALCIO 90 G,101,PyL,2026-08-01,2026-08-31
6043,LATERAL CHICO,160288000,1000037689,CREMA DENTAL COLGATE CALCIO 90 G,101,PyL,2026-08-01,2026-08-31
6630,CABECERA ANUAL FORTYLEX,162433000,1000039069,TOALLA HIGIENICA SIEMPRE LIBRE ESPECIAL CON AL...,101,PyL,2026-08-01,2026-08-31
...,...,...,...,...,...,...,...,...
6129,EXHIBIDOR COLGATE NUEVO,167101000,1000592682,CREMA DENTAL COLGATE LUMINOUS WHITE COLOR CORR...,328,PyL,2026-08-01,2026-08-31
10773,SALUS EXHIBIDOR,240513000,1000150629,SALUS FRUTTE NARANJA-DURAZNO CERO 1.5 LT,334,Bebidas,2026-08-01,2026-08-31
11162,CABECERA SALUS NEGO,240513000,1000150629,SALUS FRUTTE NARANJA-DURAZNO CERO 1.5 LT,334,Bebidas,2026-08-01,2026-08-31
10771,SALUS EXHIBIDOR,241641000,1000568930,FRUTTE SIN AZUCAR MANGO NARANJA 1.5 LT,334,Bebidas,2026-08-01,2026-08-31


In [17]:
credentials_snowflake = get_credentials("snow")

In [18]:
user, cursor, snow = snowflake_login(
                                    user = credentials_snowflake['USER'],
                                    password = credentials_snowflake['PASS'],
                                    account = credentials_snowflake['ACCOUNT']
                                    )

Intento 1
Correct Password - connected to SNOWFLAKE


In [19]:
fecha_inicio = df_total['FECHA_INICIO'].values[0]
cursor.execute("SELECT * FROM SANDBOX_PLUS.DWH.INPUT_PUNTERAS WHERE FECHA_INICIO = '" + fecha_inicio + "';")
df_old = cursor.fetch_pandas_all()
df_old

,PUNTERA,ESTADISTICO,ITEM,DESCRIPCION,LOCAL,FECHA_INICIO,FECHA_FIN,SECCION


In [22]:
df_total['ITEM'].unique()

array(['1000055471', '1000055202', '1000302904', '1000055475',
       '1000052874', '1000052863', '1000052739', '1000056037',
       '1000056040', '1000056038', '1000056039', '1000054784',
       '1000054905', '1000345459', '1000055345', '1000436504',
       '1000599319', '1000599321', '1000052960', '1000053769',
       '1000102658', '1000053752', '1000053754', '1000053763',
       '1000052980', '1000454086', '1000053371', '1000053370',
       '1000052527', '1000435994', '1000581269', '1000581284',
       '1000581270', '1000055028', '1000055034', '1000055635',
       '1000055633', '1000055843', '1000055634', '1000054975',
       '1000055865', '1000055807', '1000055808', '1000055868',
       '1000055723', '1000055016', '1000300270', '1000055744',
       '1000436723', '1000052323', '1000050441', '1000598821',
       '1000052921', '1000368339', '1000114827', '1000056104',
       '1000056111', '1000436794', '1000436795', '1000447139',
       '1000056217', '1000460962', '1000056218', '10003

In [21]:
success = carga_snow_generic(df_total.astype({'ESTADISTICO':'int64', 'ITEM':'int64', 'LOCAL':'int64'}),
                             ctx=snow, database='SANDBOX_PLUS', table='INPUT_PUNTERAS', schema='DWH')

ValueError: invalid literal for int() with base 10: 'SHAMPOO DOVE UV REPAIR 200 ML': Error while type casting for column 'ITEM'